# 50 — CUDA / GPU cross-check

Run this **before** the `00` / `1x` notebooks on a new machine. It does not generate signals or filters. It only answers:

1. Is an NVIDIA driver present (`nvidia-smi`)?
2. Did this kernel import a **CUDA** build of PyTorch (not the CPU wheel)?
3. Does `torch.cuda.is_available()` see a device?
4. Can we allocate a tensor on `cuda:0` and run a real kernel?

Any failed check raises `RuntimeError` so `nbconvert --execute` fails closed.

## 1. NVIDIA driver

In [1]:
from __future__ import annotations

import shutil
import subprocess
import sys

print(f"Python {sys.version}")
print(f"executable: {sys.executable}")

smi = shutil.which("nvidia-smi")
if smi is None:
    raise RuntimeError("nvidia-smi is not on PATH — no NVIDIA driver in this environment.")

proc = subprocess.run([smi], check=False, text=True, capture_output=True)
print(proc.stdout)
if proc.returncode != 0:
    raise RuntimeError(
        "nvidia-smi failed (driver not loaded or not visible to this process).\n"
        + proc.stderr
    )

query = subprocess.run(
    [
        smi,
        "--query-gpu=name,driver_version,compute_cap,memory.total",
        "--format=csv,noheader",
    ],
    check=True,
    text=True,
    capture_output=True,
)
print("query:", query.stdout.strip())

Python 3.12.3 (main, Nov  6 2025, 13:44:16) [GCC 13.3.0]
executable: /projects/ControlsMachineLearning/.venv/bin/python
Sun Sep 13 19:48:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060        On  |   00000000:01:00.0  On |                  N/A |
| 71%   72C    P0            158W /  170W |    6256MiB /  12288MiB |     99%      Default |
|                   

## 2. PyTorch CUDA build

In [2]:
import torch

print(f"torch {torch.__version__}")
print(f"torch.version.cuda = {torch.version.cuda}")
print(f"built with CUDA: {torch.backends.cuda.is_built()}")

ver = torch.__version__
major_minor = ver.split("+")[0].split(".")
if (int(major_minor[0]), int(major_minor[1])) < (2, 14):
    raise RuntimeError(f"Need torch >= 2.14, got {ver}")

if torch.version.cuda is None:
    raise RuntimeError(
        "This is a CPU-only PyTorch wheel (torch.version.cuda is None). "
        "Reinstall from the CUDA index in pyproject.toml (uv sync --extra gpu)."
    )

torch 2.14.0+cu132
torch.version.cuda = 13.2
built with CUDA: True


## 3. Device detection

In [3]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "torch.cuda.is_available() is False. CUDA-built torch cannot see a GPU "
        "(wrong wheel, missing driver, or this process cannot open /dev/nvidia*)."
    )

n = torch.cuda.device_count()
print(f"device_count = {n}")
for i in range(n):
    props = torch.cuda.get_device_properties(i)
    major, minor = torch.cuda.get_device_capability(i)
    print(
        f"  [{i}] {torch.cuda.get_device_name(i)}  "
        f"sm_{major}{minor}  "
        f"{props.total_memory / 1024**3:.2f} GiB"
    )

device = torch.device("cuda:0")
print(f"using {device}  current = {torch.cuda.current_device()}")

device_count = 1
  [0] NVIDIA GeForce RTX 3060  sm_86  11.60 GiB
using cuda:0  current = 0


## 4. Live kernel on `cuda:0`

A version string is not enough. Allocate, multiply, and synchronize so a missing device fails here.

In [4]:
torch.cuda.synchronize()
a = torch.randn(1024, 1024, device=device, dtype=torch.float32)
b = torch.randn(1024, 1024, device=device, dtype=torch.float32)
c = a @ b
torch.cuda.synchronize()

if c.device.type != "cuda":
    raise RuntimeError(f"matmul result landed on {c.device}, expected cuda")

print(f"a.device = {a.device}   c.device = {c.device}   c[0, 0] = {c[0, 0].item():.4f}")
print(f"allocated = {torch.cuda.memory_allocated(0) / 1024**2:.1f} MiB")
del a, b, c
torch.cuda.empty_cache()

a.device = cuda:0   c.device = cuda:0   c[0, 0] = 3.3736
allocated = 20.1 MiB


## 5. Verdict

In [5]:
print("PASS")
print(f"  GPU     {torch.cuda.get_device_name(0)}")
print(f"  driver  visible via nvidia-smi")
print(f"  torch   {torch.__version__}")
print(f"  CUDA    {torch.version.cuda}")
print(f"  device  {device}")
print("CUDA is installed and the GPU is detected. Continue with notebook 00.")

PASS
  GPU     NVIDIA GeForce RTX 3060
  driver  visible via nvidia-smi
  torch   2.14.0+cu132
  CUDA    13.2
  device  cuda:0
CUDA is installed and the GPU is detected. Continue with notebook 00.
